# Introspection Factorization — gate runner (A100 80 GB)

Runs the whole chain built so far: **G0** assets, **G1** lens validation, **G2**
injection harness, **G3** concept vectors + baseline, the **Phase 4 sweep** with
**G4a** integrity, and the **Phase 5 cascade** with **G4**. Each gate emits a
structured report and stops. Nothing prints PASS or FAIL — you read the numbers
and decide whether the next phase proceeds.

**What to expect on a cold A100 80 GB**

| step | time | GPU | note |
|---|---|---|---|
| install | ~2 min | | |
| model download | 20–40 min | | 55.6 GB, **first run only** |
| G0 | ~10 min | yes | model + 63 lens layers, per-layer SVD |
| G1 | ~25 min | yes | 80 items x 63 layers x 3 lenses |
| G2 | ~15 min | yes | invariants, dose-response, generation |
| `01_concept_vectors` | ~20 min | yes | pilot over the full pool, then extraction |
| G3 | ~30 min | yes | detection grid + controls + generation |
| `02_sweep` | ~10 min | yes | 4,320 cells → 432 forwards |
| `03_generate` | ~25 min | yes | the expensive channel, operating point only |
| G4a | seconds | **no** | audits artifacts |
| `04_factors` | ~10 min | yes | lens readouts + position control |
| G4 | seconds | **no** | the cascade |

Every stage is a separate process and loads the model itself (~3 min from cache).
That is deliberate: every artifact is a file on disk, so any stage can be re-run
alone after a disconnect. `01_concept_vectors`, `02_sweep` and `03_generate` are
additionally resumable *within* themselves. **G4a and G4 need no GPU**, so you can
re-run them any time to see what landed.

**Before you start:** the reports are the deliverable. They are written to
`artifacts/*/*_report.txt` as well as printed, so a disconnect does not lose them.

## 1 — Measure the GPU you actually have

Do not proceed on an assumption. Colab's High-RAM setting increases *system* RAM, not VRAM.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

import subprocess
out = subprocess.run(["nvidia-smi","--query-gpu=memory.total","--format=csv,noheader,nounits"],
                     capture_output=True, text=True).stdout.strip().splitlines()
vram_gb = int(out[0]) / 1024 if out and out[0].strip().isdigit() else 0
print(f"\nVRAM detected: {vram_gb:.1f} GiB")
print("Tier A (target) needs ~62 GiB: 55.6 weights + 6.6 lens fp32 + activations.")
if vram_gb < 70:
    print("!! Below the Tier A assumption. G0 will still run and report actual")
    print("!! headroom, but G1/G2 at batch 8 may OOM. Lower --batch, or subsample")
    print("!! layers with --layers-stride.")

## 2 — Install

`jlens` is pinned to the commit the sprint was verified against. `diptest` is the one dependency beyond the build spec's list — see `ASSETS.md` for why.

In [ ]:
%pip -q install "transformers>=4.57.1" accelerate huggingface_hub numpy pandas scipy scikit-learn matplotlib pyyaml diptest
%pip -q install anthropic   # only used if ANTHROPIC_API_KEY is set (LLM judge)
%pip -q install "git+https://github.com/anthropics/jacobian-lens@581d398613e5602a5af361e1c34d3a92ea82ba8e"

import torch, transformers, jlens, diptest
print("torch       ", torch.__version__, "| cuda", torch.cuda.is_available())
print("transformers", transformers.__version__)
print("jlens       ", jlens.__file__)

## 3 — Point at the repo

Upload the `introspection-factorization` folder to the runtime (or `git clone` it)
and set `REPO` below. The cell verifies every file the gates need.

In [ ]:
import os, sys, json
from pathlib import Path

REPO = Path("/content/introspection-factorization")   # <-- edit if yours differs
if not REPO.exists():
    for candidate in (Path.cwd(), Path.cwd().parent,
                      Path("/content/drive/MyDrive/introspection-factorization")):
        if (candidate / "gates" / "g0_assets.py").exists():
            REPO = candidate
            break

required = [
    "configs/sprint.yaml", "configs/baseline_words.json",
    "configs/concepts.json", "configs/judge_rubrics.json",
    "src/stats.py", "src/lens.py", "src/inject.py", "src/vectors.py",
    "src/prompts.py", "src/judge.py", "src/sweep.py", "src/factors.py",
    "gates/g0_assets.py", "gates/g1_lens.py", "gates/g2_inject.py",
    "gates/g3_baseline.py", "gates/g4a_sweep.py",
    "gates/g4_factors.py",
    "scripts/00_verify_dip.py", "scripts/01_concept_vectors.py",
    "scripts/02_sweep.py", "scripts/03_generate.py",
    "scripts/04_factors.py",
]
missing = [f for f in required if not (REPO / f).exists()]
print("REPO =", REPO)
if missing:
    raise SystemExit(f"missing files: {missing}\nSet REPO to the repo root.")
print("all", len(required), "required files present")

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
(REPO / "artifacts").mkdir(exist_ok=True)

pool = json.load(open("configs/concepts.json"))
print(f"concept pool: {pool['n']} words, source categories validated = {pool['category_counts_validated']}")
rub = json.load(open("configs/judge_rubrics.json"))
print(f"judge rubrics: {len(rub['criteria'])} vendored verbatim from eval_utils.py")

## 4 — Preflight: the statistics, on CPU

`src/stats.py` underpins every gate, so check it before spending an hour of GPU time.
This reproduces the dip comparison recorded in `ASSETS.md`: a hand-rolled dip
(wrong — it decouples the two segments at the mode), an independent linear program,
and the reference `diptest`. LP and diptest must agree; the hand-rolled one will not.

In [ ]:
!python scripts/00_verify_dip.py

In [ ]:
import numpy as np, stats
rng = np.random.default_rng(0)
print("wilson(50,100)        ", stats.wilson(50,100))
print("wilson(0,10)          ", stats.wilson(0,10), " <- stays in [0,1]")
print("participation_ratio   ", round(stats.participation_ratio(np.ones(8)),3), "(flat 8-spectrum -> 8)")
print("dip, uniform n=60     ", {k:(round(v,4) if isinstance(v,float) else v) for k,v in stats.dip_test(rng.random(60), n_boot=2000).items()})
print("dip, bimodal n=60     ", {k:(round(v,4) if isinstance(v,float) else v) for k,v in stats.dip_test(np.r_[rng.normal(0,1,30),rng.normal(8,1,30)], n_boot=2000).items()})
print("MDE base .30 n=60     ", round(stats.min_detectable_effect(0.30,60),4), " <- weak power at concept granularity")
print("MDE base .30 n=4320   ", round(stats.min_detectable_effect(0.30,4320),4))

## 5 — GATE G0: assets and environment

First run downloads the model (55.6 GB) and the lens (3.3 GB). Both are cached
afterwards.

**Three numbers to read.** `lens fitted layers` — file-size arithmetic predicts 63
of 64; this reads `source_layers` directly, and if it disagrees the memory budget
is wrong. The **identity-cosine profile** — `W_U J_l` approaches `W_U` as `l`
approaches the final layer, so the cosine should climb toward ~1.0; flat or
non-monotonic means the layer indexing is wrong, which would invalidate the layer
band and therefore the choice of 27/31/35. And **headroom**, which decides the
batch size G1/G2 can afford.

In [ ]:
!python gates/g0_assets.py 2>&1 | tee artifacts/g0_console.log

## 6 — GATE G1: lens validation

The most consequential gate. A lens that loads but does not work produces a
plausible f2 that is noise, and nothing downstream notices.

**Three numbers to read.** **Null-lens pass@1** — the random-rotation lens is the
placebo; if it scores near the Jacobian lens, the readout is coming through `W_U`
alone and the lens contributes nothing. **Median rank by layer** — where Garcia's
argument lives, and what a binary hit rate hides. **Retention counts** — if the
competence filter guts multihop, the depth-ordering cross-check loses its n.

Expect multilingual to retain poorly: its filter requires the model to emit e.g.
`pequeño` within 8 greedy tokens.

In [ ]:
!python gates/g1_lens.py --per-category 20 2>&1 | tee artifacts/g1_console.log

## 7 — GATE G2: injection harness

Most of this gate's budget goes on invariants, because they must hold exactly for
any downstream number to mean anything.

**What must be exact.** `zero-strength identity` = 0.0 (adding `0*v` is exact in
floating point, so anything else means the hook does something besides add).
`spatial containment` outside the window = 0.0 at the injection layer **and every
layer below it**. Hook fire count = 1 in prefill and 1 across `generate` — that is
the check that nothing is injected during cached decode.

**What will not be exact.** `batch equivalence` — bf16 reductions are not
order-invariant, so B=4 batched vs 4 singles differs. The raw magnitude is
reported; you judge it.

**What is a finding, not an invariant.** Leakage to positions *after* the window at
layers *above* the injection layer. 48 of 64 blocks are linear-attention with a
causal conv, so information moves forward in position. It is measured, not asserted.

In [ ]:
!python gates/g2_inject.py --layer 27 --n-concepts 10 --batch 8 2>&1 | tee artifacts/g2_console.log

## 8 — Concept vectors: pilot, selection, extraction

`scripts/01_concept_vectors.py` filters the 500-concept pool to single-token
names, measures per-concept detection over the **whole** surviving pool, selects
60 spanning the range, and extracts vectors at all three layers.

**Why it measures its own rates.** The build spec stratifies on Macar's
per-concept Gemma detection rates. Those are not published — the metrics caches
are aggregates over `(layer_idx, strength, arm)`, the abliterated checkpoint is
weights only, and the README reports aggregates. Nothing reachable carries a
per-concept number. So this measures its own, from next-token logits.

That is better in two ways and worse in one, all reported in G3: no cross-model
transfer means no regression to the mean from another model's noisy labels (the
trap the spec's own table warns about), and bimodality can be tested on the full
pool rather than on 60 concepts picked to span the range. But the cross-model
Spearman cannot be computed at all. If you have the Gemma rates, pass them as
`--stratify-file rates.json` (`{"word": rate}`) and the spec's original design
is restored.

Resumable: re-running skips the pilot and any layer already extracted.

In [ ]:
!python scripts/01_concept_vectors.py --per-tier 20 --pilot-strength 4.0 2>&1 | tail -40

In [ ]:
# what the pilot found, before G3 uses it
import json, numpy as np, sys
sys.path.insert(0, "src")
import stats

pilot = json.load(open("artifacts/vectors/pilot.json"))
rates = np.array(list(pilot["rates"].values()))
sel   = json.load(open("artifacts/vectors/selection.json"))
print(f"pilot scored {rates.size} single-token concepts at layer {pilot['layer']}, alpha {pilot['strength']}")
print(f"  mean {rates.mean():.4f}   median {np.median(rates):.4f}")
print(f"  >=0.9 : {(rates>=0.9).sum():>4}    <=0.01 : {(rates<=0.01).sum():>4}")
d = stats.dip_test(rates, n_boot=5000)
print(f"  Hartigan dip {d['dip']:.4f}  p={d['p']:.4f}   <- bimodality on the FULL pool")
a = sel["composition_audit"]
print(f"\nsingle-token filter: {a['n_before']} -> {a['n_after']}")
for b, r in a["bucket_retention"].items():
    print(f"  {b:<9} {a['bucket_before'][b]:>4} -> {a['bucket_after'].get(b,0):<4} retention {r:.3f}")
print("\na retention gap between abstract and concrete is the confound that")
print("turns tokenisation into a finding. Read it before reading G3's detection numbers.")

## 9 — GATE G3: concept vectors and baseline reproduction

**What to read.** The **composition audit** first: if abstract concepts survive
the single-token filter at a different rate than concrete ones, any
abstract/concrete difference in detection is confounded before the experiment
starts. Then **TPR − FPR** and the **AUC column** — AUC is threshold-free and
better powered, because FPR rests on only as many trials as there are task
prompts (with no injection, every concept gives identical logits, so trials
cannot be multiplied by concept). Then the **yes-bias arm**: if injecting a
concept raises P(yes) on unrelated questions whose answer is "no", part of the
detection signal is a generic yes-bias, not introspection.

**What you will not see:** any per-tier number. Concepts were selected on
measured detection, so per-tier rates would regress to the mean by construction
— the spec's own trap table forbids reporting them, and the gate does not.

In [ ]:
!python gates/g3_baseline.py --gen-layer 27 --gen-strength 4.0 2>&1 | tee artifacts/g3_console.log

### Optional: judge calibration

G3 writes 50 sampled responses to `artifacts/g3/to_label.json`. Label each
`1`/`0` for *"does this response name the injected concept?"*, save as
`artifacts/g3/hand_labels.json` in the form `{"<index>": 1, ...}`, and re-run G3
to get Cohen's kappa and the confusion matrix against the deterministic scorer.

Until then G3 prints kappa as PENDING rather than guessing at it. Set
`ANTHROPIC_API_KEY` to make the vendored LLM-judge rubrics available; without a
key the judge reports itself unavailable and never silently degrades to string
matching.

In [ ]:
import json
todo = json.load(open("artifacts/g3/to_label.json"))
print(f"{len(todo)} responses awaiting hand labels. First three:\n")
for k, v in list(todo.items())[:3]:
    print(f"[{k}] concept={v['concept']!r}\n     {v['response'][:160]!r}\n")

## 10 — The sweep

4,320 cells: 60 concepts x 3 layers x 4 strengths x 2 orders x 3 conditions.
One shard per concept, written atomically; re-running skips what exists.

**Logits-first.** Each cell caches the residual at the report position (swept
layers + final) and the JSON-boolean probabilities. Full-vocabulary logits are
never stored — they are recoverable exactly from the cached final-layer residual,
since unembed is deterministic. That is the difference between 350 MB and 4 GB.

**The report position had to be constructed.** "Next-token logits at the report
position" is unambiguous for report-first, where the model's first output token
*is* the detection verdict. For task-first the report lands inside the
generation, so reading both at the last prompt token would compare a detection
verdict against a task answer and call the difference an order effect. The
assistant turn is therefore prefilled up to the detection key in whichever order
the protocol demands, making the next token a JSON boolean in both arms.

**4,320 cells is not 4,320 forwards.** Batching 8 concepts through one forward
gives 540; deduplicating zero-strength cells gives 432. The dedupe is exact —
alpha=0 adds exactly zero, which G2 measures as the zero-strength identity — and
it is done per batch, not once globally, so G4a still has a per-batch control
statistic for the drift probe.

In [ ]:
!python scripts/02_sweep.py --batch 8 2>&1 | tail -25

## 11 — Generation at the operating point

The expensive channel, so it runs at one (layer, strength) for both orders and
all three conditions. **T=1.0 with 4 samples** — Garcia used T=0 with a single
completion and flags it as a limitation, so sampling with several draws is a
cheap, legible improvement over the closest prior work, and it is what makes f3
an estimate with a spread rather than one draw.

Resumable per batch of concepts. This is the long pole; if the runtime drops,
re-run the same cell and it picks up where it stopped.

In [ ]:
!python scripts/03_generate.py --layer 27 --strength 4.0 --samples 4 2>&1 | tail -20

## 12 — GATE G4a: sweep integrity

Audits the artifacts, not the model — **no GPU needed**, so you can re-run it
any time to see what actually landed.

**What it is for:** the failures that leave no error message. A missing cell. A
shard written twice under the wrong name (caught by hashing residual payloads —
two concepts with byte-identical residuals across every cell is not a
coincidence). A NaN that propagated. And **temporal drift**: the zero-strength
control compared between the first and last decile of shards by write time. A
systematic shift there means something changed mid-run — a reloaded model, a
different dtype, a hardware switch — and every number in the run inherits it.

Also read `P(true)+P(false)`. If that sum is not near 1, the prefill is not
landing where it was meant to and the next token is not a JSON boolean at all,
which would quietly invalidate the whole detection channel.

In [ ]:
!python gates/g4a_sweep.py 2>&1 | tee artifacts/g4a_console.log

## 13 — Factor inputs (the GPU stage of Phase 5)

`scripts/04_factors.py` turns sweep shards and generations into the arrays the
cascade needs: lens readouts from the cached residuals, and the **position
control**. The sweep caches only the report position — that is what keeps shards
at 350 MB — so the injection and random positions are recomputed here. That fresh
pass also recomputes the *report* position, giving a free consistency check
against the cache.

**Why the generator was changed.** `f₁` and `f₂` are properties of a prefill
residual; `f₃` is a property of the continuation sampled from it. If the two came
from different prompts, "the same trial" would be undefined and `cascade_residual`
would measure my own inconsistency instead of a denominator bug. So
`03_generate.py` now continues from *exactly* the prefilled prompt the sweep
cached — the model simply finishes the JSON object it was started on.

Add `--r-lens` to also read out with the R-lens from `camilablank/workspace-lenses`;
it is guarded, and reports itself unavailable rather than crashing if the file
does not match the expected format.

In [ ]:
!python scripts/04_factors.py --layer 27 --strength 4.0 2>&1 | tail -15

## 14 — GATE G4: the three-factor decomposition

    P(report) = P(represented) x P(verbalizable | represented) x P(reported | verbalizable)

No GPU. **Read `cascade_residual` first** — it should be floating-point zero at
every k and in both orders. The chain rule makes the product exact when the
denominators nest, so anything above ~1e-12 means a stage dropped trials or took
a different denominator, and every number below it is suspect. It is `nan`, not
zero, wherever nothing survives to f₃ — that is undefined, not a failure.

Then **survivorship**. If only a handful of trials reach f₃, the f₃ number is
fragile no matter how tight its interval looks; the jackknife range and the
judge-noise interval bound how much to trust it.

Then the **controls**: the probe null (shuffled labels) should collapse to the
FPR the threshold was set at — 0.05, not 0. The **f₂ null band** matters because
cosine has no absolute scale at d=5120, so f₂ only means something relative to
matched-norm random vectors through the identical pipeline. And the **position
control** — only the report position supports the claim; if injection or random
positions score similarly, the finding is a global shift, not position-specific
structure.

In [ ]:
!python gates/g4_factors.py --k 10 2>&1 | tee artifacts/g4_console.log

In [ ]:
# the cascade at a glance, straight from the artifact
import json
g4 = json.load(open("artifacts/g4/g4_factors.json"))
h = g4["headline"]
print(f"f1 (represented)               {h['f1']:.4f}")
print(f"f2 (verbalizable | represented) {h['f2']:.4f}")
print(f"f3 (reported | verbalizable)    {h['f3']:.4f}")
print(f"product                         {h['f1']*h['f2']*h['f3']:.6f}")
print(f"observed                        {h['observed_cascade_rate']:.6f}")
print(f"cascade_residual                {h['residual']:.3e}   <- must be ~0")
print()
print(f"survivorship  {h['n_entering_f1']} -> {h['n_surviving_f1']} -> "
      f"{h['n_surviving_f2']} -> {h['n_surviving_f3']}")
print(f"naive report rate               {g4['naive_report_rate']:.4f}")
print("\nlosses: representation failure, verbalizability failure, channel closure")
print(f"  1 - f1 = {1-h['f1']:.4f}   1 - f2 = {1-h['f2']:.4f}   1 - f3 = {1-h['f3']:.4f}")

## 15 — Collect the reports

Everything is on disk. Download `artifacts/` before the runtime recycles.

In [ ]:
from pathlib import Path
for p in sorted(Path("artifacts").rglob("*")):
    if p.is_file() and p.suffix != ".npz":
        print(f"{p.stat().st_size:>12,}  {p}")
shards = list(Path("artifacts/sweep").glob("shard_*.npz")) if Path("artifacts/sweep").exists() else []
print(f"\n{len(shards)} sweep shards, {sum(p.stat().st_size for p in shards)/2**20:.1f} MiB")

print("\n" + "="*70)
print("re-read any report without re-running its gate:")
for g in ("g0","g1","g2","g3","g4a","g4"):
    print(f"  print(open('artifacts/{g}/{g}_report.txt').read())")

In [ ]:
# zip the artifacts for download
!zip -qr artifacts.zip artifacts && ls -lh artifacts.zip
try:
    from google.colab import files
    files.download("artifacts.zip")
except Exception as e:
    print("not on Colab, download artifacts.zip manually:", type(e).__name__)